# Proyecto QTranslate LIFIA

## Quirk → OpenQASM 3.0 → Ecosistema objetivo

Objetivo: Este cuaderno jupyter permite la ejecución del flujo de traducción de circuitos diseñados mediante la herramienta gráfica Quirk y la ejecución de etapas complementarias de verificación (visuales, ZX calculo, etc). Actualmente permite la trasucción de circuitos hacia 3 proveedores (IBM, Amazon y Xanadu).

Flujo de ejecución:
1. Descarga e instalación de librerías externas necesarias.
2. Traducción de circuitos [Quirk](https://algassert.com/quirk) hacia OpenQASM (versión configurable 2.0+), utilizado como lenguaje de "pivote" de representación intermedia.

    Carga de los circuitos de prueba (.json) seleccionados por la configuración del `.env`: `QTRANS_INPUT_FILE` en modo `algorithm` o `QTRANS_INPUT_FILES` (o todos los `.json` de `QTRANS_INPUT_DIR`) en modo `batch`.

3. Etapa de comparativas visuales (path: `/output/run_DD_MM_YYYY__HH/imgs/`):
   1. Generación de circuitos visuales en Quirk (path: `circuits_quirk`)
   2. Generación de circuitos visuales en OpenQasm (path: `circuits_qasm`)
   3. Generación de directorio de comparación de algoritmos (path: `visual_comparations`)
   
4. Etapa de equivalencia circuital comprobada mediante ZXCalculus (path: `/output/run_DD_MM_YYYY__HH/ZXCalculus/`)
5. Migración de código python al ecosistema/lenguaje objetivo (path: `/output/run_DD_MM_YYYY__HH/migrated_circuits/`), actualmente: IBM Qiskit, AWS Braket o Xanadu Pennylane.

> Todas las salidas se agrupan en `output/run_<estampilla_temporal>/`. 
> La estampilla temporal conserva fecha y hora (con formato: DD_MM_YYYY__HH): dentro de la misma hora se reutiliza el mismo directorio y se sobrescriben los archivos generados. Con el fin de consolidar a lo sumo una prueba o corrida, por hora.


### Descarga de las librerías necesarias para la funcionalidad


In [8]:
import sys
import subprocess
import importlib.util

packages = {
    'qiskit': 'qiskit',
    'matplotlib': 'matplotlib',
    'pylatexenc': 'pylatexenc',
    'qiskit_qasm3_import': 'qiskit-qasm3-import',
    'pandas': 'pandas',
    'pyzx': 'pyzx',
}

detalle_info = ''
paquetes_instalados = 0

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        estado = 'no existente, se instaló correctamente ✓'
        paquetes_instalados += 1
    else:
        estado = 'existente previamente, no se instaló ❌'

    info = f"{pip_name} - {estado}"
    detalle_info += (', ' if detalle_info else '') + info
    print(f'Paquete {info}')

print(f'\nℹ️ {paquetes_instalados} paquetes instalados recientemente en el kernel')


Paquete qiskit - existente previamente, no se instaló ❌
Paquete matplotlib - existente previamente, no se instaló ❌
Paquete pylatexenc - existente previamente, no se instaló ❌
Paquete qiskit-qasm3-import - existente previamente, no se instaló ❌
Paquete pandas - existente previamente, no se instaló ❌
Paquete pyzx - existente previamente, no se instaló ❌

ℹ️ 0 paquetes instalados recientemente en el kernel


### Verificación de existencia y localización de circuitos de prueba (.json)

Se genera una tabla de resumen (formato: MarkDown) por cada archivo de circuitos de prueba seleccionado por la configuración del `.env`:

- `QTRANS_INPUT_MODE=algorithm`: solo el archivo indicado en `QTRANS_INPUT_FILE`.
- `QTRANS_INPUT_MODE=batch`: los archivos de `QTRANS_INPUT_FILES` (separados por comas) o, si está vacío, todos los `.json` de `QTRANS_INPUT_DIR`.

Al modificar el `.env`, basta con volver a ejecutar la celda siguiente para recalcular la selección.

In [9]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

# La raíz del proyecto se resuelve desde la ubicación raíz del proyecto.
# Así ``input/`` y ``output/`` apuntan siempre a la raíz del proyecto aunque QTrans.ipynb se ejecute desde el directorio /notebooks.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / 'notebooks')
        if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró qtrans/utils/qutils.py; ejecutá el cuadernillo dentro del proyecto')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# override=True: al editar .env alcanza con volver a ejecutar esta celda para
# recalcular el modo y los archivos de entrada seleccionados.
load_dotenv(PROJECT_ROOT / '.env', override=True)

from qtrans.utils.qutils import quirk_circuit_info, quirk_to_qasm
from qtrans.utils.pipeline_utils import format_run_label


def _env_text(name, default=''):
    """Valor de una variable QTRANS_* sin espacios sobrantes."""
    return (os.getenv(name) or default).strip()


def _env_bool(name, default=True):
    """Booleano configurado en .env (1/true/yes/on)."""
    value = _env_text(name)
    if not value:
        return default
    return value.casefold() in {'1', 'true', 'yes', 'on'}


# Interruptores de la etapa ZX (mismos nombres y defaults que el pipeline CLI).
GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)
GENERATE_ZX_IMAGES = _env_bool('QTRANS_GENERATE_ZX_IMAGES', True)


def _configured_path(name, default):
    """Ruta configurada en .env; las relativas se resuelven contra la raíz del proyecto."""
    value = _env_text(name)
    if not value:
        return Path(default).resolve()
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / str(path).lstrip('/\\')
    return path.resolve()


# Constantes de directorios de entrada y salida del proyecto (configurables en .env)
INPUT_DIR = _configured_path('QTRANS_INPUT_DIR', PROJECT_ROOT / 'input')
OUTPUT_DIR = _configured_path('QTRANS_OUTPUT_DIR', PROJECT_ROOT / 'output')
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

# Etiqueta de la corrida por fecha y hora (sin minutos): /output/run_DD_MM_YYYY__HH
# Dentro de la misma hora se reutiliza el directorio y se sobrescriben los archivos.
# Se puede personalizar con QTRANS_RUN_LABEL (tokens DD/MM/YYYY/HH/SS).
RUN_LABEL_FORMAT = _env_text('QTRANS_RUN_LABEL')
RUN_LABEL = (
    format_run_label(RUN_LABEL_FORMAT)
    if RUN_LABEL_FORMAT
    else datetime.now().strftime('%d_%m_%Y__%H')
)

# Rutas de salida para la corrida actual (siempre en la raíz del proyecto). La
# estructura de carpetas NO se crea aquí: cada celda crea, bajo demanda, el
# directorio que necesita o genera (con Path.mkdir(parents=True, exist_ok=True)).
RUN_DIR = OUTPUT_DIR / f'run_{RUN_LABEL}'
RUN_QASM_DIR = RUN_DIR / 'algorithms_qasm'
RUN_IMGS_DIR = RUN_DIR / 'imgs'
RUN_CIRCUITS_DIR = RUN_DIR / 'migrated_circuits'
RUN_ZX_DIR = RUN_DIR / 'ZXCalculus'
RUN_COMPARE_DIR = RUN_IMGS_DIR / 'visual_comparations'

print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR}')
print(f"ESTAMPA (fecha y hora): {RUN_LABEL}  -> durante esta hora se reutiliza el directorio y se sobrescriben los archivos.")

# Selección de circuitos de entrada, equivalente a la del pipeline CLI:
# - algorithm: solo QTRANS_INPUT_FILE
# - batch: QTRANS_INPUT_FILES (lista separada por comas) o todos los *.json de QTRANS_INPUT_DIR
INPUT_MODE = _env_text('QTRANS_INPUT_MODE', 'batch').casefold()
if INPUT_MODE not in {'algorithm', 'batch'}:
    raise ValueError(f"QTRANS_INPUT_MODE debe ser 'algorithm' o 'batch' (recibido: {INPUT_MODE!r})")
INPUT_FILE = _env_text('QTRANS_INPUT_FILE') or None
INPUT_FILES = tuple(
    name.strip() for name in _env_text('QTRANS_INPUT_FILES').split(',') if name.strip()
)


def _resolve_input_file(raw_name):
    """Resuelve un nombre de .json contra el directorio de entrada configurado."""
    candidate = Path(raw_name).expanduser()
    if not candidate.is_absolute():
        candidate = INPUT_DIR / candidate
    candidate = candidate.resolve()
    if candidate.suffix.lower() != '.json':
        raise ValueError(f'Los archivos de entrada deben tener extensión .json: {candidate}')
    if not candidate.is_file():
        raise FileNotFoundError(f'No se encontró el archivo de entrada configurado: {candidate}')
    return candidate


def resolve_selected_input_files():
    """Archivos .json que se procesarán, según QTRANS_INPUT_MODE y QTRANS_INPUT_FILE(S)."""
    if INPUT_MODE == 'algorithm':
        if INPUT_FILE is None:
            raise ValueError("QTRANS_INPUT_FILE es obligatorio cuando QTRANS_INPUT_MODE='algorithm'")
        if INPUT_FILES:
            raise ValueError("QTRANS_INPUT_FILES solo se utiliza cuando QTRANS_INPUT_MODE='batch'")
        return [_resolve_input_file(INPUT_FILE)]

    if INPUT_FILE is not None:
        raise ValueError(
            "QTRANS_INPUT_FILE solo se utiliza cuando QTRANS_INPUT_MODE='algorithm'; "
            "dejá QTRANS_INPUT_FILE= vacío para que el modo batch use QTRANS_INPUT_FILES "
            "o todos los .json del directorio de entrada"
        )
    if INPUT_FILES:
        return sorted({_resolve_input_file(name) for name in INPUT_FILES})
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f'No existe el directorio de entrada configurado: {INPUT_DIR}')
    selected_files = sorted(INPUT_DIR.glob('*.json'))
    if not selected_files:
        raise FileNotFoundError(f'No se encontraron archivos .json en {INPUT_DIR}')
    return selected_files


FILE_LABELS = {
    'main_algorithms': 'Algoritmos principales',
    'popular_algorithms': 'Algoritmos populares',
    'qft_algorithms': 'Algoritmos QFT',
}
FILE_PREFIXES = {
    'main_algorithms': 'main_',
    'popular_algorithms': 'popular_',
    'qft_algorithms': 'qft_',
}

SELECTED_INPUT_FILES = resolve_selected_input_files()
ALGORITHM_COLLECTIONS = []
for input_path in SELECTED_INPUT_FILES:
    with input_path.open('r', encoding='utf-8') as handle:
        algorithms = json.load(handle)
    if not isinstance(algorithms, dict):
        raise ValueError(f'El archivo de entrada debe contener un objeto JSON: {input_path}')
    ALGORITHM_COLLECTIONS.append({
        'path': input_path,
        'stem': input_path.stem,
        'label': FILE_LABELS.get(input_path.stem, input_path.stem),
        'prefix': FILE_PREFIXES.get(input_path.stem, f'{input_path.stem}_'),
        'algorithms': algorithms,
    })


def find_input_dir():
    """Directorio de entrada configurado (QTRANS_INPUT_DIR), sin depender del cwd."""
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f'No existe el directorio de entrada configurado: {INPUT_DIR}')
    return INPUT_DIR


def load_algorithms(name):
    """Devuelve los algoritmos de un archivo seleccionado por .env (por su stem)."""
    for collection in ALGORITHM_COLLECTIONS:
        if collection['stem'] == name:
            return collection['algorithms']
    selected = ', '.join(collection['stem'] for collection in ALGORITHM_COLLECTIONS)
    raise ValueError(f'La fuente de algoritmos {name!r} no está seleccionada por .env; activas: {selected}')


def _selected_collection(name):
    """Compatibilidad: colección seleccionada por nombre o dict vacío si no aplica."""
    return next(
        (collection['algorithms'] for collection in ALGORITHM_COLLECTIONS if collection['stem'] == name),
        {},
    )


# Variables de compatibilidad usadas por las celdas siguientes.
ALGORITHMS = _selected_collection('main_algorithms')
POPULAR_ALGORITHMS = _selected_collection('popular_algorithms')
QFT_ALGORITHMS = _selected_collection('qft_algorithms')


def describe_algorithms(collection):
    """Muestra en Markdown la tabla de circuitos de un archivo de entrada seleccionado."""
    algorithms = collection['algorithms']
    relative_source = os.path.relpath(collection['path'], PROJECT_ROOT).replace(os.sep, '/')
    markdown = (
        f"## {collection['label']}\n\n"
        f"Archivo: `{relative_source}`\n\n"
        '| # | Circuito | Descripción | # Qubits |\n'
        '|---|---|---|---:|\n'
    )
    for index, (key, value) in enumerate(algorithms.items(), 1):
        url = value.get('url') if isinstance(value, dict) else None
        if not isinstance(url, str):
            raise ValueError(f'El algoritmo {key!r} de {collection["path"].name} debe contener una URL Quirk')
        info = quirk_circuit_info(url)
        algorithm_name = key.split('. ', 1)[1] if '. ' in key else key
        number = key.split('. ', 1)[0] if '. ' in key else str(index)
        description = str(value.get('desc', '')).replace('|', r'\|')
        markdown += f'| {number} | **{algorithm_name}** | {description} | {info["n_qubits"]} |\n'
    display(Markdown(markdown))
    print(f'ℹ️ Se detectaron {len(algorithms)} circuitos de entrada en {collection["path"].name}.')


print(f'MODO DE ENTRADA (QTRANS_INPUT_MODE): {INPUT_MODE}')
print(f'DIRECTORIO DE ENTRADA (QTRANS_INPUT_DIR): {INPUT_DIR}')
if INPUT_MODE == 'algorithm':
    print(f'ARCHIVO DE ENTRADA (QTRANS_INPUT_FILE): {INPUT_FILE}')
else:
    print(f'ARCHIVOS DE ENTRADA (QTRANS_INPUT_FILES): {", ".join(path.name for path in SELECTED_INPUT_FILES)}')

for collection in ALGORITHM_COLLECTIONS:
    describe_algorithms(collection)

total_algoritmos = sum(len(collection['algorithms']) for collection in ALGORITHM_COLLECTIONS)
print(
    f'\n✓ Total de circuitos a procesar según .env: {total_algoritmos} '
    f'en {len(ALGORITHM_COLLECTIONS)} archivo(s).'
)
print(
    f'ZX (QTRANS_GENERATE_ZX): {GENERATE_ZX}  |  '
    f'IMÁGENES ZX (QTRANS_GENERATE_ZX_IMAGES): {GENERATE_ZX_IMAGES}'
)


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09
ESTAMPA (fecha y hora): 23_10_2026__09  -> durante esta hora se reutiliza el directorio y se sobrescriben los archivos.
MODO DE ENTRADA (QTRANS_INPUT_MODE): algorithm
DIRECTORIO DE ENTRADA (QTRANS_INPUT_DIR): C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\input
ARCHIVO DE ENTRADA (QTRANS_INPUT_FILE): custom_gates_algorithms.json


## custom_gates_algorithms

Archivo: `input/custom_gates_algorithms.json`

| # | Circuito | Descripción | # Qubits |
|---|---|---|---:|
| 1 | **complex_customs** | Circuito complejo diseñado ad-hoc | 2 |
| 2 | **rotational_customs** | Circuito con una compuerta custom utilizada y 8 definidas | 2 |


ℹ️ Se detectaron 2 circuitos de entrada en custom_gates_algorithms.json.

✓ Total de circuitos a procesar según .env: 2 en 1 archivo(s).


### Captura de screenshots de circuitos originales Quirk

Generación de archivos de circuitos en el path: `.../imgs/circuits_quirk`


In [10]:
import subprocess
import sys

try:
    import selenium  # noqa: F401
except ImportError:
    print("Instalando selenium...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'selenium'])

from pathlib import Path
import re

from qtrans.utils.qutils import capture_quirk_circuit
from qtrans.utils.pipeline_utils import safe_algorithm_name

visual_root = RUN_IMGS_DIR

quirk_original_dir = visual_root / 'circuits_quirk'
quirk_original_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

def natural_sort_key(value):
    text = str(value).strip()
    match = re.search(r'^(\d+)', text)
    if match:
        num = int(match.group(1))
        return (num, text)
    else:
        return (float('inf'), text)

print('Captura de screenshots de circuitos originales desde Quirk\n')

if 'ALGORITHM_COLLECTIONS' not in globals():
    raise RuntimeError(
        'Falta la selección de circuitos (.env): ejecutá primero la celda '
        '"Verificación de existencia y localización de circuitos de prueba".'
    )

captured = 0
total_circuits = sum(len(collection['algorithms']) for collection in ALGORITHM_COLLECTIONS)
algorithm_collections = [
    (collection['algorithms'], collection['prefix'])
    for collection in ALGORITHM_COLLECTIONS
]

# Recorre la colección de algoritmos
for algorithms, filename_prefix in algorithm_collections:
    for name, data in sorted(algorithms.items(), key=lambda item: natural_sort_key(item[0])):
        url = data['url']
        safe_name = safe_algorithm_name(name)
        output_path = quirk_original_dir / f'{filename_prefix}{safe_name}.png'

        print(f'Capturando: {name}...', end=' ')
        if capture_quirk_circuit(url, output_path):
            print(f'✅ Guardado en {output_path.name}')
            captured += 1
        else:
            print('❌ Error')

print(f'\n✅ Etapa de capturas completada: {captured}/{total_circuits} capturas de circuitos generadas')
print(f'\nℹ️ Archivos disponibles en: {quirk_original_dir}')


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09
Captura de screenshots de circuitos originales desde Quirk

Capturando: 1. complex_customs... ✅ Guardado en custom_gates_algorithms_1__complex_customs.png
Capturando: 2. rotational_customs... ✅ Guardado en custom_gates_algorithms_2__rotational_customs.png

✅ Etapa de capturas completada: 2/2 capturas de circuitos generadas

ℹ️ Archivos disponibles en: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\imgs\circuits_quirk


# Traducción de circuitos a OpenQASM


### Generación de circuitos OpenQASM

Traducción de circuitos Quirk --> OpenQASM
En formato textual (archivos .txt)

La versión de OpenQASM puede ser: 3.0 o 2.0 dependiendo de la parametrización.

Archivos generados en el path: `.../algorithms_qasm`


In [11]:
import os
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

from qtrans.utils.pipeline_utils import safe_algorithm_name

# Versión de OpenQASM a generar: configurable en .env (QTRANS_QASM_VERSION);
# por defecto '3.0', acepta '2.0'.
dotenv_path = (globals().get('PROJECT_ROOT') or Path.cwd()) / '.env'
load_dotenv(dotenv_path, override=True)
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or "3.0").strip()
if QASM_VERSION not in {'2.0', '3.0'}:
    raise ValueError(f"QASM_VERSION debe ser '2.0' o '3.0' (recibido: {QASM_VERSION!r})")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'


def _env_bool(name, default=True):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


# Verbosidad y activación del workflow ZX (mismos parámetros que el pipeline CLI).
VERBOSE = _env_bool('QTRANS_VERBOSE', True)
GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)
GENERATE_ZX_IMAGES = _env_bool('QTRANS_GENERATE_ZX_IMAGES', True)


def strip_qasm_version(stem):
    """Quita el sufijo _vX.Y del nombre; las celdas posteriores lo usan para
    derivar los artefactos (imágenes, grafos, scripts) sin el sufijo de versión."""
    return re.sub(r'_v\d+\.\d+$', '', stem)


# Directorio versionado por corrida: /output/run_DD_MM_YYYY__HH/algorithms_qasm
output_dir = RUN_QASM_DIR
output_dir.mkdir(parents=True, exist_ok=True)
# Elimina archivos de corridas/ejecuciones anteriores en el mismo directorio
# para evitar procesar circuitos que ya no están seleccionados por .env.
for old_file in output_dir.glob('*.txt'):
    try:
        old_file.unlink()
    except Exception:
        pass
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
print(f'VERSIÓN DE OpenQASM: {QASM_VERSION}')
print(f'ZX: {"activado" if GENERATE_ZX else "desactivado"} (QTRANS_GENERATE_ZX)')
print(f'Imágenes ZX: {"activadas" if GENERATE_ZX_IMAGES else "desactivadas"} (QTRANS_GENERATE_ZX_IMAGES)')


if 'ALGORITHM_COLLECTIONS' not in globals():
    raise RuntimeError(
        'Falta la selección de circuitos (.env): ejecutá primero la celda '
        '"Verificación de existencia y localización de circuitos de prueba".'
    )


def generate_qasm_table_and_files():
    rows = []
    generated_paths = []
    algorithm_collections = [
        (collection["label"], collection["algorithms"], collection["prefix"])
        for collection in ALGORITHM_COLLECTIONS
    ]

    for collection_name, algorithms, filename_prefix in algorithm_collections:
        for name, data in algorithms.items():
            url = data["url"]
            info = quirk_circuit_info(url)
            qasm_code = quirk_to_qasm(url, data.get("offset", 0), qasm_version=QASM_VERSION)

            safe_name = safe_algorithm_name(name)
            file_path = output_dir / f"{filename_prefix}{safe_name}{QASM_VERSION_SUFFIX}.txt"
            file_path.write_text(qasm_code, encoding="utf-8")
            generated_paths.append(file_path.resolve())

            rows.append({
                "Colección": collection_name,
                "Algoritmo": name.split(". ", 1)[1] if ". " in name else name,
                "Qubits": info["n_qubits"],
                "Columnas": info["n_cols"],
                "Descripcion": data.get("desc", ""),
                "Archivo": str(file_path.name),
                "Líneas": len(qasm_code.splitlines())
            })

    total_archivos = len(generated_paths)
    print(f"\n ✅ Todos los archivos generados en el path: {output_dir.resolve()} ({total_archivos} archivos)")
    for generated_path in generated_paths:
        print(f"- {generated_path.name}")

    df = pd.DataFrame(rows)
    return df


generate_qasm_table_and_files()


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09
VERSIÓN DE OpenQASM: 3.0

 ✅ Todos los archivos generados en el path: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\algorithms_qasm (2 archivos)
- custom_gates_algorithms_1__complex_customs_v3.0.txt
- custom_gates_algorithms_2__rotational_customs_v3.0.txt


,Colección,Algoritmo,Qubits,Columnas,Descripcion,Archivo,Líneas
0,custom_gates_algorithms,complex_customs,2,6,Circuito complejo diseñado ad-hoc,custom_gates_algorithms_1__complex_customs_v3....,64
1,custom_gates_algorithms,rotational_customs,2,2,Circuito con una compuerta custom utilizada y ...,custom_gates_algorithms_2__rotational_customs_...,52


# Verificación visual de traducción de circuitos


### Generación de imágenes de circuitos traducidos OpenQASM3

Genera archivos en el path: `.../imgs/circuits_qasm`


In [12]:
from pathlib import Path
import re

from qtrans.utils.qutils import render_qasm_image

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)

visual_root = RUN_IMGS_DIR


circuits_dir = visual_root / 'circuits_qasm'
circuits_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [(0, int(p)) if p.isdigit() else (1, p.lower()) for p in parts if p]


def render_qasm_circuits():
    qasm_dir = RUN_QASM_DIR
    circuits_dir = visual_root / 'circuits_qasm'
    circuits_dir.mkdir(parents=True, exist_ok=True)

    if not qasm_dir.exists():
        raise FileNotFoundError(f'No existe la carpeta de QASM: {qasm_dir}')

    qasm_version_suffix = globals().get('QASM_VERSION_SUFFIX', '')
    files = sorted((p for p in qasm_dir.glob('*.txt') if not qasm_version_suffix or p.name.endswith(qasm_version_suffix + '.txt')), key=lambda p: natural_sort_key(p.stem))
    if not files:
        raise FileNotFoundError(f'No hay archivos .txt en {qasm_dir}')

    rendered = 0
    failed = 0
    for txt_path in files:
        qasm_text = txt_path.read_text(encoding='utf-8')
        png_path = circuits_dir / f'{strip_qasm_version(txt_path.stem)}.png'
        try:
            # Carga + dibujo unificados (qtrans.utils.qutils.render_qasm_image).
            render_qasm_image(qasm_text, output_path=png_path, style='bw')
            print(f'Generado: {png_path.name}')
            rendered += 1
        except Exception as exc:
            print(f'No se pudo renderizar {txt_path.name} ({type(exc).__name__}): {exc!r}')
            failed += 1

    print(f'\nℹ️ Resumen: {rendered}/{len(files)} imágenes generadas; {failed} archivos con error.')


print(f'\nℹ️ Generación de imágenes de circuitos traducidos a OpenQASM 3.0 ( {circuits_dir.resolve()} )\n')

render_qasm_circuits()


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09

ℹ️ Generación de imágenes de circuitos traducidos a OpenQASM 3.0 ( C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\imgs\circuits_qasm )

Generado: custom_gates_algorithms_1__complex_customs.png
Generado: custom_gates_algorithms_2__rotational_customs.png

ℹ️ Resumen: 2/2 imágenes generadas; 0 archivos con error.


### Comparación visual

Generación del subdirectorio para agilizar la verificación de equivalencia visual y la exactitud de la traducción de circuitos.
Establece la generación de imágenes generadas (circuitos quirk y OpenQASM)

Directorio de salida: `.../compare`


In [13]:
from pathlib import Path
import os
import re

compare_dir = RUN_COMPARE_DIR
compare_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

visual_root = RUN_IMGS_DIR

qasm_dir = visual_root / 'circuits_qasm'
quirk_dir = visual_root / 'circuits_quirk'
qasm_rel = os.path.relpath(qasm_dir, compare_dir).replace(os.sep, '/')
quirk_rel = os.path.relpath(quirk_dir, compare_dir).replace(os.sep, '/')

if not qasm_dir.exists():
    raise FileNotFoundError(f'No existe la carpeta de circuitos OpenQASM: {qasm_dir}')
if not quirk_dir.exists():
    raise FileNotFoundError(f'No existe la carpeta de circuitos Quirk: {quirk_dir}')


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [(0, int(part)) if part.isdigit() else (1, part.lower()) for part in parts if part]


qasm_pngs = {path.stem: path for path in qasm_dir.glob('*.png')}
quirk_pngs = {path.stem: path for path in quirk_dir.glob('*.png')}
matching_names = sorted(set(qasm_pngs) & set(quirk_pngs), key=natural_sort_key)

index_lines = ['# Comparación de circuitos', '']

for stem in matching_names:
    qasm_path = qasm_pngs[stem]
    quirk_path = quirk_pngs[stem]
    title = stem.replace('_', ' ')
    md_path = compare_dir / f'{stem}.md'
    md_path.write_text(
        '\n'.join([
            f'# Comparación: {title}',
            '',
            '<table>',
            '<tr><th align="center">Circuito Original (Quirk)</th><th align="center">Circuito traducido (OpenQASM 3.0)</th></tr>',
            '<tr>',
            '<td align="center"><a href="' + quirk_rel + '/' + quirk_path.name + '" title="Abrir imagen original"><img src="' + quirk_rel + '/' + quirk_path.name + '" alt="' + quirk_path.name + '" width="100%" /></a></td>',
            '<td align="center"><a href="' + qasm_rel + '/' + qasm_path.name + '" title="Abrir imagen original"><img src="' + qasm_rel + '/' + qasm_path.name + '" alt="' + qasm_path.name + '" width="100%" /></a></td>',
            '</tr>',
            '</table>',
            '',
        ]),
        encoding='utf-8'
    )

    index_lines.append(f'- [{title}]({stem}.md)')
    print(f'Generado info comparativa --> {md_path.name}')

index_path = compare_dir / 'README.md'
index_path.write_text('\n'.join(index_lines) + '\n', encoding='utf-8')

print(f'\nℹ️ Carpeta de comparación creada en: {compare_dir.resolve()}')
print(f'\nℹ️ Se generaron {len(matching_names)} comparativas para archivos presentes en ambas carpetas.')
print(f'\nℹ️ Índice disponible en: {index_path.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09
Generado info comparativa --> custom_gates_algorithms_1__complex_customs.md
Generado info comparativa --> custom_gates_algorithms_2__rotational_customs.md

ℹ️ Carpeta de comparación creada en: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\imgs\visual_comparations

ℹ️ Se generaron 2 comparativas para archivos presentes en ambas carpetas.

ℹ️ Índice disponible en: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\imgs\visual_comparations\README.md


# Verificación de equivalencia entre circuitos mediante ZXCalculus


### Generación de grafos ZX a partir del código Quirk (importado a un .json)

Directorio asociado: `.../ZXCalculus/algorithms_base`

Si un circuito no es representable en ZX (por ejemplo, una matriz custom 4×4), la celda informa el motivo y continúa con el resto; no aborta la generación.

Esta celda solo se ejecuta si `QTRANS_GENERATE_ZX=true`.


In [15]:
import json
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = globals().get('PROJECT_ROOT') or next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('No se encontró qtrans/utils/qutils.py desde el directorio del notebook')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Relee .env en cada ejecución para que los interruptores reflejen el archivo actual.
load_dotenv(Path(project_root) / '.env', override=True)


def _env_bool(name, default=True):
    """Booleano configurado en .env (1/true/yes/on)."""
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)

import importlib
from qtrans.utils import qutils, zx_utils
importlib.reload(qutils)
importlib.reload(zx_utils)
quirk_to_zx_graph = zx_utils.quirk_to_zx_graph

if not GENERATE_ZX:
    print('ℹ️ Generación de grafos ZX base omitida: QTRANS_GENERATE_ZX=false.')
else:
    if 'ALGORITHM_COLLECTIONS' not in globals():
        raise RuntimeError(
            'Falta la selección de circuitos (.env): ejecutá primero la celda '
            '"Verificación de existencia y localización de circuitos de prueba".'
        )

    algorithm_collections = [
        (collection['algorithms'], collection['prefix'])
        for collection in ALGORITHM_COLLECTIONS
    ]
    # Reutiliza la configuración de la corrida; si la celda se ejecuta aislada, la reconstruye
    if 'RUN_ZX_DIR' not in globals():
        from datetime import datetime
        def _find_output_root():
            _root = globals().get('PROJECT_ROOT') or Path.cwd()
            for _candidate in (_root / 'output', Path.cwd() / 'output', Path.cwd().parent / 'output'):
                if _candidate.is_dir():
                    return _candidate
            return _root / 'output'
        _output_root = _find_output_root()
        _existing_runs = sorted(
            (d for d in _output_root.glob('run_*') if d.is_dir()),
            key=lambda d: d.stat().st_mtime,
            reverse=True,
        )
        RUN_DIR = _existing_runs[0] if _existing_runs else _output_root / f"run_{datetime.now().strftime('%d_%m_%Y__%H')}"
        RUN_ZX_DIR = RUN_DIR / 'ZXCalculus'
    output_dir = RUN_ZX_DIR / 'algorithms_base'
    print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
    quirk_time = 0.5

    graph_files = {}
    # Circuitos que no admiten representación ZX (p. ej. matrices 4x4): son una
    # limitación conocida de la etapa, no un fallo de la corrida. Se informan y se
    # continúa con el resto de los algoritmos.
    ZX_BASE_SKIPPED = {}
    for collection, filename_prefix in algorithm_collections:
        for name, data in collection.items():
            safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')
            filename = f'{filename_prefix}{safe_name}.json'
            if filename in graph_files or filename in ZX_BASE_SKIPPED:
                raise ValueError(f'Nombre de archivo duplicado: {filename}')
            try:
                graph = quirk_to_zx_graph(data['url'], data.get('offset', 0), time=quirk_time)
                graph_json = graph.to_json()
                json.loads(graph_json)
            except Exception as exc:
                reason = f'{type(exc).__name__}: {exc}'
                ZX_BASE_SKIPPED[filename] = reason
                print(f'⚠️ Sin grafo ZX para {name}: {reason}')
                continue
            graph_files[filename] = graph_json

    expected_graphs = sum(len(collection) for collection, _ in algorithm_collections)
    if len(graph_files) + len(ZX_BASE_SKIPPED) != expected_graphs:
        raise RuntimeError(
            f'Se esperaban {expected_graphs} algoritmos, se procesaron '
            f'{len(graph_files) + len(ZX_BASE_SKIPPED)}'
        )

    if graph_files:
        output_dir.mkdir(parents=True, exist_ok=True)
        for filename, graph_json in graph_files.items():
            (output_dir / filename).write_text(graph_json, encoding='utf-8')

    print(f'Grafo ZX generado para {len(graph_files)}/{expected_graphs} algoritmos.')
    if ZX_BASE_SKIPPED:
        print(f'ℹ️ {len(ZX_BASE_SKIPPED)} circuitos sin grafo ZX (no representables en ZX):')
        for filename, reason in sorted(ZX_BASE_SKIPPED.items()):
            print(f'- {filename.removesuffix(".json")}: {reason}')
    print(f'Snapshot temporal de Quirk: {quirk_time} (aplica a gates dependientes del tiempo como Rxft).')
    if graph_files:
        print(f'Archivos guardados en: {output_dir.resolve()}')
    else:
        print(f'Sin grafos base que guardar en: {output_dir.resolve()}')


ℹ️ Generación de grafos ZX base omitida: QTRANS_GENERATE_ZX=false.


### Generación de grafos ZX a partir del código OpenQasm generado a partir de nuestra migración

Directorio asociado: `.../ZXCalculus/algorithms_qasm`

Esta celda procesa únicamente los archivos `.txt` de la versión activa (`QASM_VERSION`, por defecto 3.0), identificada por el sufijo `_v3.0.txt` / `_v2.0.txt`.

Si un circuito no es representable en ZX, la celda informa el motivo y continúa con el resto; no aborta la generación.

Esta celda solo se ejecuta si `QTRANS_GENERATE_ZX=true`.


In [16]:
import importlib
import json
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = globals().get('PROJECT_ROOT') or next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('No se encontró qtrans/utils/qutils.py desde el directorio del notebook')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Relee .env en cada ejecución para que los interruptores reflejen el archivo actual.
load_dotenv(Path(project_root) / '.env', override=True)


def _env_bool(name, default=True):
    """Booleano configurado en .env (1/true/yes/on)."""
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)

from qtrans.utils import qutils, zx_utils
importlib.reload(qutils)
importlib.reload(zx_utils)
qasm_to_zx_graph = zx_utils.qasm_to_zx_graph

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)

if not GENERATE_ZX:
    print('ℹ️ Generación de grafos ZX desde OpenQASM omitida: QTRANS_GENERATE_ZX=false.')
else:
    # Reutiliza la configuración de la corrida; si la celda se ejecuta aislada, la reconstruye
    if 'RUN_QASM_DIR' not in globals() or 'RUN_ZX_DIR' not in globals():
        from datetime import datetime
        def _find_output_root():
            _root = globals().get('PROJECT_ROOT') or Path.cwd()
            for _candidate in (_root / 'output', Path.cwd() / 'output', Path.cwd().parent / 'output'):
                if _candidate.is_dir():
                    return _candidate
            return _root / 'output'
        _output_root = _find_output_root()
        _existing_runs = sorted(
            (d for d in _output_root.glob('run_*') if (d / 'algorithms_qasm').is_dir()),
            key=lambda d: d.stat().st_mtime,
            reverse=True,
        )
        RUN_DIR = _existing_runs[0] if _existing_runs else _output_root / f"run_{datetime.now().strftime('%d_%m_%Y__%H')}"
        RUN_QASM_DIR = RUN_DIR / 'algorithms_qasm'
        RUN_ZX_DIR = RUN_DIR / 'ZXCalculus'
    qasm_dir = RUN_QASM_DIR
    output_dir = RUN_ZX_DIR / 'algorithms_qasm'
    print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
    qasm_version_suffix = globals().get('QASM_VERSION_SUFFIX', '')
    qasm_files = sorted(p for p in qasm_dir.glob('*.txt') if not qasm_version_suffix or p.name.endswith(qasm_version_suffix + '.txt'))
    print(f'VERSIÓN DE OpenQASM procesada: {qasm_version_suffix or "(todos)"}')
    if not qasm_files:
        raise FileNotFoundError(f'No hay archivos .txt para la versión activa {qasm_version_suffix!r} en {qasm_dir}')

    graph_files = {}
    # Circuitos cuyo OpenQASM no admite representación ZX: se informan y se continúa.
    ZX_QASM_SKIPPED = {}
    for qasm_path in qasm_files:
        filename = f'{strip_qasm_version(qasm_path.stem)}.json'
        if filename in graph_files or filename in ZX_QASM_SKIPPED:
            raise ValueError(f'Nombre de archivo duplicado: {filename}')
        qasm_code = qasm_path.read_text(encoding='utf-8')
        try:
            graph = qasm_to_zx_graph(qasm_code)
            graph_json = graph.to_json()
            json.loads(graph_json)
        except Exception as exc:
            reason = f'{type(exc).__name__}: {exc}'
            ZX_QASM_SKIPPED[filename] = reason
            print(f'⚠️ Sin grafo ZX para {qasm_path.name}: {reason}')
            continue
        graph_files[filename] = graph_json

    expected_graphs = len({strip_qasm_version(p.stem) for p in qasm_files})
    if len(graph_files) + len(ZX_QASM_SKIPPED) != expected_graphs:
        raise RuntimeError(
            f'Se esperaban {expected_graphs} algoritmos, se procesaron '
            f'{len(graph_files) + len(ZX_QASM_SKIPPED)}'
        )

    if graph_files:
        output_dir.mkdir(parents=True, exist_ok=True)
        for filename, graph_json in graph_files.items():
            (output_dir / filename).write_text(graph_json, encoding='utf-8')

    print(f'Grafos ZX generados: {len(graph_files)}/{expected_graphs}.')
    if ZX_QASM_SKIPPED:
        print(f'ℹ️ {len(ZX_QASM_SKIPPED)} circuitos sin grafo ZX (no representables en ZX):')
        for filename, reason in sorted(ZX_QASM_SKIPPED.items()):
            print(f'- {filename.removesuffix(".json")}: {reason}')
    print(f'Circuitos OpenQASM leídos desde: {qasm_dir.resolve()}')
    if graph_files:
        print(f'Grafos guardados en: {output_dir.resolve()}')
    else:
        print(f'Sin grafos OpenQASM que guardar en: {output_dir.resolve()}')


ℹ️ Generación de grafos ZX desde OpenQASM omitida: QTRANS_GENERATE_ZX=false.


### Verificación de Equivalencia - Comparación de grafos ZX (Quirk vs. OpenQASM)

Establecemos por cada algoritmo si la comparación de equivalencia es correcta, a fin de verificar la migración hacia OpenQASM.

Los circuitos que no pudieron representarse en ZX se muestran como **NO COMPARABLES** con el motivo correspondiente.

Esta celda solo se ejecuta si `QTRANS_GENERATE_ZX=true`.


In [17]:
import os
import re
import sys
from pathlib import Path

import pyzx as zx
from dotenv import load_dotenv
from IPython.display import Markdown, display

from qtrans.utils.zx_utils import compare_zx_graphs

project_root = globals().get('PROJECT_ROOT') or next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()),
    None,
)
if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if project_root is not None:
    # Relee .env en cada ejecución para que el interruptor refleje el archivo actual.
    load_dotenv(Path(project_root) / '.env', override=True)


def _env_bool(name, default=True):
    """Booleano configurado en .env (1/true/yes/on)."""
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)


def normalized_graph_name(path):
    return re.sub(r'_+', '_', path.stem).casefold()


def algorithm_order(path):
    stem = path.stem.casefold()
    is_popular = stem.startswith('popular_')
    name = stem.removeprefix('popular_') if is_popular else stem
    match = re.match(r'(\d+)', name)
    number = int(match.group(1)) if match else float('inf')
    return (is_popular, number, name)


def index_graphs(directory):
    """Indexa los grafos JSON por nombre normalizado; {} si la carpeta no existe o está vacía."""
    if not directory.is_dir():
        return {}
    indexed = {}
    for path in sorted(directory.glob('*.json')):
        key = normalized_graph_name(path)
        if key in indexed:
            raise ValueError(f'Nombres ambiguos en {directory}: {indexed[key].name} y {path.name}')
        indexed[key] = path
    return indexed


def markdown_escape(value):
    return str(value).replace('|', '\\|').replace('\n', ' ')


if not GENERATE_ZX:
    print('ℹ️ Comparación ZX omitida: QTRANS_GENERATE_ZX=false.')
else:
    zx_dir = RUN_ZX_DIR
    print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

    base_dir = zx_dir / 'algorithms_base'
    qasm_dir = zx_dir / 'algorithms_qasm'

    base_graphs = index_graphs(base_dir)
    qasm_graphs = index_graphs(qasm_dir)
    all_keys = sorted(
        set(base_graphs) | set(qasm_graphs),
        key=lambda key: algorithm_order(Path(f'{key}.json')),
    )
    if not all_keys:
        print(f'⚠️ No hay grafos ZX para comparar en {base_dir} ni en {qasm_dir}.')
    else:
        results = []
        for key in all_keys:
            base_path = base_graphs.get(key)
            qasm_path = qasm_graphs.get(key)
            reference_path = base_path or qasm_path
            if base_path is None or qasm_path is None:
                if base_path is None and qasm_path is None:
                    reason = 'Sin grafos ZX base ni OpenQASM'
                elif base_path is None:
                    reason = 'Sin grafo ZX base (circuito no representable en ZX)'
                else:
                    reason = 'Sin grafo ZX de OpenQASM (circuito no representable en ZX)'
                results.append({
                    'Nombre algoritmo': reference_path.stem,
                    'Método utilizado': 'sin grafo',
                    'Resultado': 'NO COMPARABLES',
                    'Motivo': reason,
                })
                continue

            base_graph = zx.Graph.from_json(base_path.read_text(encoding='utf-8'))
            qasm_graph = zx.Graph.from_json(qasm_path.read_text(encoding='utf-8'))

            # Veredicto unificado: misma regla de equivalencia que el pipeline CLI.
            method, result = compare_zx_graphs(base_graph, qasm_graph)

            results.append({
                'Nombre algoritmo': base_path.stem,
                'Método utilizado': method,
                'Resultado': result,
                'Motivo': '',
            })

        comparison_md = '| Circuito | Método | Resultado | Motivo |\n|---|---|---|---|\n'
        for row in results:
            comparison_md += (
                f"| {markdown_escape(row['Nombre algoritmo'])} "
                f"| {markdown_escape(row['Método utilizado'])} "
                f"| {markdown_escape(row['Resultado'])} "
                f"| {markdown_escape(row['Motivo'])} |\n"
            )
        display(Markdown(comparison_md))


ℹ️ Comparación ZX omitida: QTRANS_GENERATE_ZX=false.


### Generación de imágenes (grafos) asociados a los algoritmos base

Toma cada algoritmo presente en el directorio `/algorithms_base` y genera un .svg para comprobar la reducción.
Directorio de salida: `/ZXCalculus/graphs`

Esta celda solo se ejecuta si `QTRANS_GENERATE_ZX=true` y `QTRANS_GENERATE_ZX_IMAGES=true`.


In [ ]:
import os
import re
import sys
import warnings
from pathlib import Path

import pyzx as zx
from dotenv import load_dotenv
from IPython.display import Markdown, display

from qtrans.utils.zx_utils import reduce_and_draw_zx_svg

zx.settings.drawing_backend = 'matplotlib'

project_root = globals().get('PROJECT_ROOT') or next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'qtrans' / 'utils' / 'qutils.py').is_file()),
    None,
)
if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if project_root is not None:
    # Relee .env en cada ejecución para que los interruptores reflejen el archivo actual.
    load_dotenv(Path(project_root) / '.env', override=True)


def _env_bool(name, default=True):
    """Booleano configurado en .env (1/true/yes/on)."""
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)
GENERATE_ZX_IMAGES = _env_bool('QTRANS_GENERATE_ZX_IMAGES', True)


def graph_order(path):
    stem = path.stem.casefold()
    is_popular = stem.startswith('popular_')
    name = stem.removeprefix('popular_') if is_popular else stem
    match = re.match(r'(\d+)', name)
    number = int(match.group(1)) if match else float('inf')
    return is_popular, number, name


def markdown_escape(value):
    return str(value).replace('|', '\\|').replace('\n', ' ')


if not GENERATE_ZX:
    print('ℹ️ Generación de imágenes ZX omitida: QTRANS_GENERATE_ZX=false.')
elif not GENERATE_ZX_IMAGES:
    print('ℹ️ Generación de imágenes ZX omitida: QTRANS_GENERATE_ZX_IMAGES=false.')
else:
    zx_dir = RUN_ZX_DIR
    print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

    notebook_dir = globals().get('NOTEBOOK_DIR') or (globals().get('PROJECT_ROOT') or Path.cwd()) / 'notebooks'

    base_dir = zx_dir / 'algorithms_base'
    output_dir = zx_dir / 'graphs'
    graph_paths = sorted(base_dir.glob('*.json')) if base_dir.is_dir() else []
    if not graph_paths:
        print(f'⚠️ No hay grafos base ZX en {base_dir}; no se generan SVGs.')

    ordered_graph_paths = sorted(graph_paths, key=graph_order)
    image_results = []
    failed_count = 0

    if ordered_graph_paths:
        output_dir.mkdir(parents=True, exist_ok=True)
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive')
            for index, graph_path in enumerate(ordered_graph_paths, start=1):
                graph = zx.Graph.from_json(graph_path.read_text(encoding='utf-8'))
                n_qubits = graph.num_inputs()
                vertices_before = graph.num_vertices()
                image_path = output_dir / f'{graph_path.stem}.svg'
                reduced_graph = graph

                try:
                    # Reducción + render unificados (mismo helper que el pipeline CLI).
                    reduced_graph = reduce_and_draw_zx_svg(graph, image_path)
                    status = 'Generado'
                except Exception as exc:
                    failed_count += 1
                    status = f'Error: {type(exc).__name__}'
                vertices_after = reduced_graph.num_vertices()

                relative_path = Path(os.path.relpath(image_path, start=notebook_dir)).as_posix()
                image_reference = f'[Abrir SVG]({relative_path})' if image_path.is_file() else '-'
                image_results.append({
                    'Nombre algoritmo': graph_path.stem,
                    'Cúbits': n_qubits,
                    'Vértices (antes → después)': f'{vertices_before} → {vertices_after}',
                    'Referencia al grafo': image_reference,
                    'Resultado': status,
                })

        headers = ['Nombre algoritmo', 'Cúbits', 'Vértices (antes → después)', 'Referencia al grafo', 'Resultado']
        markdown_rows = [
            '| ' + ' | '.join(headers) + ' |',
            '| ' + ' | '.join('---' for _ in headers) + ' |',
        ]
        for row in image_results:
            cells = [
                markdown_escape(row['Nombre algoritmo']),
                str(row['Cúbits']),
                row['Vértices (antes → después)'],
                row['Referencia al grafo'],
                markdown_escape(row['Resultado']),
            ]
            markdown_rows.append('| ' + ' | '.join(cells) + ' |')

        display(Markdown('\n'.join(markdown_rows)))
        print(f'\nSVG generados: {len(image_results) - failed_count}/{len(ordered_graph_paths)}; errores: {failed_count}.')
        print(f'Archivos guardados en: {output_dir.resolve()}')


# Migración de circuitos a ecosistemas / lenguajes concretos


### Generación de circuitos cuánticos a partir del código OpenQASM

Dos pasos, en celdas separadas:

1. **Carga**: lee los `.txt` de `output/run_DD_MM_YYYY__HH/algorithms_qasm` y hace el `loads()` del OpenQASM por proveedor con `cargar_circuito_qasm()` (`qtrans/utils/qutils.py`), dejando los circuitos en el diccionario `circuitos_cargados`.
2. **Generación**: a partir de esa carga, `provider_python_source()` crea un script `.py` por circuito y proveedor.

Los proveedores salen del `.env`: `QTRANS_TARGET_PROVIDERS`, con valores canónicos `qiskit`, `pennylane` y `braket` (alias: `ibm`, `aws`, `xanadu`).

Salida: `output/run_DD_MM_YYYY__HH/migrated_circuits/<subcarpeta>/`, con `ibm_qiskit/`, `aws_braket/` y `xanadu_pennylane/`.

In [18]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from qtrans.utils.qutils import PROVIDER_DIRECTORY, cargar_circuito_qasm, providers_from_env

if 'RUN_QASM_DIR' not in globals():
    raise RuntimeError('Faltan las rutas de la corrida: ejecutá primero la celda de configuración (PROJECT_ROOT / RUN_*).')

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)


def _env_flag(name, default=True):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or globals().get('QASM_VERSION') or '3.0').strip()
if QASM_VERSION not in ('2.0', '3.0'):
    raise ValueError(f"QASM_VERSION no válido: {QASM_VERSION!r} (use '2.0' o '3.0')")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'

SHOTS = int(os.getenv('QTRANS_SHOTS') or 1000)
PRESERVE_CUSTOM_GATES = _env_flag('QTRANS_PRESERVE_CUSTOM_GATES', True)

# Proveedores del .env: QTRANS_TARGET_PROVIDERS (qiskit, pennylane, braket).
PROVIDERS = providers_from_env()

qasm_dir = RUN_QASM_DIR
if not qasm_dir.is_dir():
    # Respaldo: si esta corrida aún no tiene OpenQASM, se leen los archivos del
    # run más reciente que sí los tenga (se avisa para no abortar por la hora).
    candidatos = sorted(
        (
            candidate / 'algorithms_qasm'
            for candidate in RUN_DIR.parent.glob('run_*')
            if (candidate / 'algorithms_qasm').is_dir()
        ),
        key=lambda candidate: candidate.parent.stat().st_mtime,
        reverse=True,
    )
    if candidatos:
        qasm_dir = candidatos[0]
        print(
            f'⚠️ No hay OpenQASM en {RUN_DIR.name}; '
            f'se leen los archivos de {qasm_dir.parent.name}.'
        )
    else:
        raise FileNotFoundError(
            f'No existe la carpeta de OpenQASM: {RUN_QASM_DIR} (ni en ningún run_*). '
            'Ejecutá antes la celda "Generación de circuitos OpenQASM".'
        )

qasm_files = sorted(
    (
        path for path in qasm_dir.iterdir()
        if path.is_file() and path.suffix.lower() == '.txt'
        and path.name.endswith(QASM_VERSION_SUFFIX + '.txt')
    ),
    key=lambda path: path.name,
)
if not qasm_files:
    raise FileNotFoundError(
        f'No se encontraron archivos OpenQASM {QASM_VERSION} '
        f'(*{QASM_VERSION_SUFFIX}.txt) en {qasm_dir}'
    )

qasm_sources = {
    strip_qasm_version(path.stem): path.read_text(encoding='utf-8')
    for path in qasm_files
}

print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
print(f'VERSIÓN DE OpenQASM procesada: QASM {QASM_VERSION}')
print(f'PROVEEDORES: {", ".join(PROVIDERS)}')

# loads() del OpenQASM por proveedor (qtrans.utils.qutils.cargar_circuito_qasm).
circuitos_cargados = {provider: {} for provider in PROVIDERS}
carga_filas = []
carga_errores = []

for provider in PROVIDERS:
    for name, qasm_source in qasm_sources.items():
        try:
            circuito = cargar_circuito_qasm(
                qasm_source,
                provider,
                qasm_version=QASM_VERSION,
                shots=SHOTS,
                preserve_custom_gates=PRESERVE_CUSTOM_GATES,
            )
        except Exception as exc:
            carga_errores.append((provider, name, exc))
            print(f'No se pudo cargar {provider}/{name} ({type(exc).__name__}): {exc!r}')
            continue
        circuitos_cargados[provider][name] = circuito
        carga_filas.append({
            'Proveedor': provider,
            'Circuito': name,
            'Tipo': type(circuito).__name__,
        })

df_carga = pd.DataFrame(carga_filas, columns=['Proveedor', 'Circuito', 'Tipo'])
display(df_carga)

print('\nℹ️ Circuitos cargados por proveedor:')
for provider in PROVIDERS:
    cargados = len(circuitos_cargados[provider])
    subcarpeta = PROVIDER_DIRECTORY.get(provider, provider)
    estado = 'OK' if cargados == len(qasm_sources) else 'INCOMPLETO'
    print(f'- {provider} (migrated_circuits/{subcarpeta}): '
          f'{cargados}/{len(qasm_sources)} circuitos [{estado}]')

if carga_errores:
    print(f'\n⚠️ {len(carga_errores)} circuitos no se pudieron cargar (detalle arriba).')
print(f'\nℹ️ {len(qasm_sources)} archivos OpenQASM leídos desde: {qasm_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09
VERSIÓN DE OpenQASM procesada: QASM 3.0
PROVEEDORES: qiskit


,Proveedor,Circuito,Tipo
0,qiskit,custom_gates_algorithms_1__complex_customs,QuantumCircuit
1,qiskit,custom_gates_algorithms_2__rotational_customs,QuantumCircuit
2,qiskit,zx_flag_test_input_1__Bernstein-Vazirani,QuantumCircuit



ℹ️ Circuitos cargados por proveedor:
- qiskit (migrated_circuits/ibm_qiskit): 3/3 circuitos [OK]

ℹ️ 3 archivos OpenQASM leídos desde: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\algorithms_qasm


In [19]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from qtrans.utils.qutils import PROVIDER_DIRECTORY, provider_python_source

if 'RUN_CIRCUITS_DIR' not in globals():
    raise RuntimeError('Faltan las rutas de la corrida: ejecutá primero la celda de configuración (PROJECT_ROOT / RUN_*).')
if 'qasm_sources' not in globals() or 'circuitos_cargados' not in globals():
    raise RuntimeError('Falta la carga previa: ejecutá la celda anterior (cargar_circuito_qasm).')
if 'PROVIDERS' not in globals():
    raise RuntimeError('Falta la definición de PROVIDERS: ejecutá la celda anterior.')

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)


def _env_flag(name, default=True):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or globals().get('QASM_VERSION') or '3.0').strip()
if QASM_VERSION not in ('2.0', '3.0'):
    raise ValueError(f"QASM_VERSION no válido: {QASM_VERSION!r} (use '2.0' o '3.0')")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'

SHOTS = int(os.getenv('QTRANS_SHOTS') or 1000)
PRESERVE_CUSTOM_GATES = _env_flag('QTRANS_PRESERVE_CUSTOM_GATES', True)

migrated_root = RUN_CIRCUITS_DIR
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

generacion_filas = []
generacion_errores = []

for provider in PROVIDERS:
    if provider not in circuitos_cargados:
        continue
    subcarpeta = PROVIDER_DIRECTORY.get(provider, provider)
    provider_dir = migrated_root / subcarpeta
    provider_dir.mkdir(parents=True, exist_ok=True)

    for name, qasm_source in qasm_sources.items():
        if name not in circuitos_cargados[provider]:
            # Solo se migra lo que la celda anterior cargó para ese proveedor.
            continue
        try:
            python_source = provider_python_source(
                qasm_source,
                provider,
                qasm_version=QASM_VERSION,
                shots=SHOTS,
                preserve_custom_gates=PRESERVE_CUSTOM_GATES,
            )
            output_path = provider_dir / f'{name}.py'
            compile(python_source, str(output_path), 'exec')
            output_path.write_text(python_source, encoding='utf-8')
        except Exception as exc:
            generacion_errores.append((provider, name, exc))
            print(f'Error generando {provider}/{name} ({type(exc).__name__}): {exc!r}')
            continue

        generacion_filas.append({
            'Proveedor': provider,
            'Script': f'{subcarpeta}/{output_path.name}',
            'Líneas': len(python_source.splitlines()),
        })

df_generacion = pd.DataFrame(generacion_filas, columns=['Proveedor', 'Script', 'Líneas'])
display(df_generacion)

print('\nℹ️ Scripts migrados por proveedor:')
for provider in PROVIDERS:
    subcarpeta = PROVIDER_DIRECTORY.get(provider, provider)
    generados = sum(1 for fila in generacion_filas if fila['Proveedor'] == provider)
    print(f'- {provider} -> migrated_circuits/{subcarpeta}: {generados} scripts')

if generacion_errores:
    print(f'\n⚠️ {len(generacion_errores)} scripts con error (detalle arriba).')
print(f'\nℹ️ Carpeta de circuitos migrados: {migrated_root.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09


,Proveedor,Script,Líneas
0,qiskit,ibm_qiskit/custom_gates_algorithms_1__complex_...,83
1,qiskit,ibm_qiskit/custom_gates_algorithms_2__rotation...,59
2,qiskit,ibm_qiskit/zx_flag_test_input_1__Bernstein-Vaz...,25



ℹ️ Scripts migrados por proveedor:
- qiskit -> migrated_circuits/ibm_qiskit: 3 scripts

ℹ️ Carpeta de circuitos migrados: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\migrated_circuits


### Generación de circuitos utilizando IBM Composer
#### Celda opcional de validación visual de circuitos cuánticos migrados

Recorre los algoritmos migrados en `migrated_circuits/ibm_qiskit` y, a partir de su código OpenQASM, genera en `migrated_circuits/ibm_qiskit/imgs` una imagen `.png` por circuito con el estilo de IBM Quantum Composer (render local con Qiskit, `circuit.draw(output='mpl', style='iqp')`).

In [20]:
from pathlib import Path
import os
import re
from dotenv import load_dotenv

from qtrans.utils.qutils import render_qasm_image

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)

# Reutiliza la configuración de la corrida; si la celda se ejecuta aislada, la reconstruye
if 'RUN_CIRCUITS_DIR' not in globals() or 'RUN_QASM_DIR' not in globals():
    from datetime import datetime

    def _find_output_root():
        _root = globals().get('PROJECT_ROOT') or Path.cwd()
        for _candidate in (_root / 'output', Path.cwd() / 'output', Path.cwd().parent / 'output'):
            if _candidate.is_dir():
                return _candidate
        return _root / 'output'

    _output_root = _find_output_root()
    _existing_runs = sorted(
        (
            d for d in _output_root.glob('run_*')
            if (d / 'migrated_circuits' / 'ibm_qiskit').is_dir()
            or (d / 'migrated_circuits' / 'qiskit').is_dir()  # corridas anteriores
        ),
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    RUN_DIR = _existing_runs[0] if _existing_runs else _output_root / f"run_{datetime.now().strftime('%d_%m_%Y__%H')}"
    RUN_QASM_DIR = RUN_DIR / 'algorithms_qasm'
    RUN_CIRCUITS_DIR = RUN_DIR / 'migrated_circuits'

load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or globals().get('QASM_VERSION') or '3.0').strip()
if QASM_VERSION not in ('2.0', '3.0'):
    raise ValueError(f"QASM_VERSION no válido: {QASM_VERSION!r} (use '2.0' o '3.0')")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'

qasm_dir = RUN_QASM_DIR
# Los scripts de Qiskit viven en migrated_circuits/ibm_qiskit (nombre legado: 'qiskit').
qiskit_subdir = 'ibm_qiskit' if (RUN_CIRCUITS_DIR / 'ibm_qiskit').is_dir() else 'qiskit'
qiskit_dir = RUN_CIRCUITS_DIR / qiskit_subdir
images_dir = qiskit_dir / 'imgs'
images_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

if not qiskit_dir.is_dir():
    raise FileNotFoundError(
        f'No existe la carpeta de circuitos migrados a Qiskit: {qiskit_dir}. '
        'Ejecutá antes la celda de generación y verificá que QTRANS_TARGET_PROVIDERS incluya qiskit.'
    )


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [(0, int(part)) if part.isdigit() else (1, part.lower()) for part in parts if part]


migrated_scripts = sorted(qiskit_dir.glob('*.py'), key=lambda path: natural_sort_key(path.stem))
if not migrated_scripts:
    raise FileNotFoundError(f'No hay scripts migrados (.py) en {qiskit_dir}')

rendered = 0
failed = 0
for script_path in migrated_scripts:
    name = strip_qasm_version(script_path.stem)
    qasm_path = qasm_dir / f'{name}{QASM_VERSION_SUFFIX}.txt'
    if not qasm_path.is_file():
        print(f'Sin OpenQASM asociado para {script_path.name}: {qasm_path.name}')
        failed += 1
        continue

    png_path = images_dir / f'{name}.png'
    try:
        # Dibujo unificado con el estilo de IBM Quantum Composer.
        render_qasm_image(
            qasm_path.read_text(encoding='utf-8'),
            QASM_VERSION,
            style='iqp',
            output_path=png_path,
        )
        print(f'Generado: {png_path.name}')
        rendered += 1
    except Exception as exc:
        print(f'No se pudo dibujar {script_path.name} ({type(exc).__name__}): {exc!r}')
        failed += 1

print(f'\nℹ️ Resumen: {rendered}/{len(migrated_scripts)} imágenes generadas; {failed} archivos con error.')
print(f'Imágenes (estilo IBM Quantum Composer) guardadas en: {images_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09
Generado: custom_gates_algorithms_1__complex_customs.png
Generado: custom_gates_algorithms_2__rotational_customs.png
Generado: zx_flag_test_input_1__Bernstein-Vazirani.png

ℹ️ Resumen: 3/3 imágenes generadas; 0 archivos con error.
Imágenes (estilo IBM Quantum Composer) guardadas en: C:\Users\JoséManuel Suarez\desarrollo\QTransLIFIA\output\run_23_10_2026__09\migrated_circuits\ibm_qiskit\imgs
